# Agricultural Crop Yield Forecasting with Random Forest

This tutorial notebook demonstrates the complete workflow for predicting agricultural crop yields using **Random Forest Regression**.

### Outline:
1. **Load Agro-Climatic Dataset** (Yield, Rainfall, Temperatures, Heat Stress Days, NDVI, Soil Moisture)
2. **Feature Engineering** (Autoregressive Lags & Rolling Window Weather Averages)
3. **Cross-Validation** using expanding-window `TimeSeriesSplit` to prevent future-data leakage
4. **Model Training & Evaluation** (RMSE, MAE, MAPE, R-squared)
5. **Interpretability** (Mean Decrease in Impurity & Partial Dependence Plots)
6. **Interactive Visualizations** with Plotly


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import PartialDependenceDisplay

# Load sample agricultural dataset
df = pd.read_csv('../data/sample_yield_weather.csv')
df.head(10)


## 1. Feature Engineering for Time Series

Create autoregressive yield lags ($Y_{t-1}, Y_{t-2}$) and 2-year rolling weather averages.


In [ ]:
df = df.sort_values('year').reset_index(drop=True)
df['yield_lag1'] = df['yield_kg_ha'].shift(1)
df['yield_lag2'] = df['yield_kg_ha'].shift(2)
df['rainfall_ma2'] = df['rainfall_mm'].rolling(2).mean()
df['temp_ma2'] = df['temp_mean_c'].rolling(2).mean()

model_data = df.dropna().reset_index(drop=True)
print('Prepared dataset shape:', model_data.shape)
model_data.head()


## 2. Walk-Forward Cross-Validation

We use `TimeSeriesSplit` to respect temporal ordering.


In [ ]:
features = [
    'yield_lag1', 'yield_lag2', 'rainfall_mm', 'temp_max_c',
    'temp_mean_c', 'heat_stress_days', 'solar_rad_mj',
    'ndvi_peak', 'soil_moisture_pct', 'rainfall_ma2', 'temp_ma2'
]
target = 'yield_kg_ha'

X = model_data[features].astype(float)
y = model_data[target].astype(float)

tscv = TimeSeriesSplit(n_splits=5)
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    rf = RandomForestRegressor(n_estimators=150, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    print(f'Fold {fold} | Test Years: {model_data.loc[test_idx, "year"].values} | RMSE: {rmse:.1f} kg/ha | MAPE: {mape:.2f}%')


## 3. Fit Final Model and Variable Importance


In [ ]:
final_rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
final_rf.fit(X, y)

imp = pd.Series(final_rf.feature_importances_, index=features).sort_values(ascending=False)
fig_imp = px.bar(
    x=imp.values, y=imp.index, orientation='h',
    title='Relative Feature Importance (Mean Decrease in Impurity)',
    labels={'x': 'Gini Importance Score', 'y': 'Predictor Variable'},
    color=imp.values, color_continuous_scale='Viridis'
)
fig_imp.update_layout(yaxis=dict(autorange='reversed'), template='plotly_white')
fig_imp.show()


## 4. Interactive Actual vs. Forecasted Trajectory


In [ ]:
all_preds = final_rf.predict(X)
fig_traj = go.Figure()
fig_traj.add_trace(go.Scatter(x=model_data['year'], y=y, mode='lines+markers', name='Observed Yield', line=dict(color='#065f46', width=2.5)))
fig_traj.add_trace(go.Scatter(x=model_data['year'], y=all_preds, mode='lines+markers', name='RF Forecast', line=dict(color='#0284c7', width=2, dash='dash')))
fig_traj.update_layout(
    title='Historical Observed vs. Random Forest Predicted Crop Yield',
    xaxis_title='Cropping Season',
    yaxis_title='Yield (kg/ha)',
    template='plotly_white'
)
fig_traj.show()
